<h1> Causal disovery modular experiments batteries

<h2> Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio
from scipy.io import loadmat
from Preprocessing.imbalanced_performance.cleaned_preprocess import ModularProcesses, CommonProcesses
from project_methods import ProjectMethods
from pathlib import Path
from causal_discovery.algos.notears import NoTears
import networkx as nx
import hashlib
import os
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.GraphUtils import GraphUtils
import matplotlib.image as mpimg
import algorithms.dotears as dotears

<h2> Dataset

In [ ]:
groups = [[1, 4, 6], [2, 7], [3, 8], [5]]
cols = ['Step_Index', 'Ambient_TemperatureC', 'VoltageV', 'CurrentA', 'Cycle_Index', 'TemperatureC_Cell_1', 'TemperatureC_Cell_2', 'TemperatureC_Cell_3', 'TemperatureC_Cell_4','CurrentA_Cell_1', 'CurrentA_Cell_2', 'CurrentA_Cell_3','CurrentA_Cell_4']
experiment_dataset = ModularProcesses.getExperimentDataSet()
multicausaldataset = CommonProcesses.multienvcausaldiscoverydataset(experiment_dataset, cols, groups)
multicausaldatasetdivided = dict(tuple(multicausaldataset.groupby("Group")))

In [ ]:
print(multicausaldataset.head())

In [ ]:
print(multicausaldataset.shape)
print(multicausaldataset.isnull().sum())

<h2> Causal Discovery

In [ ]:
labels = [multicausaldataset.columns]
# Convert to numpy array
data = multicausaldataset.to_numpy()
print(data.shape)
print(labels)
n_samples = data.shape[0]
n_features = data.shape[1]
pc_result = pc(data)

<h3> NoTears

<h4> Per group

In [ ]:
dags = ProjectMethods.NoTearsByGroup(multicausaldatasetdivided, '../Src/structures/3modular_level_2/images/multienv/NoTears/ByGroup')
print(dags)

In [ ]:
dag_list, cols = ProjectMethods.load_all_dags("../Src/structures/3modular_level_2/images/multienv/NoTears/ByGroup")           # from saved files
freq_matrix = ProjectMethods.dag_frequency_matrix(dag_list)
print("Matrix shape:", freq_matrix.shape)
print("Non-zero elements:", np.count_nonzero(freq_matrix))
print(freq_matrix)
ProjectMethods.plot_dag_frequency_heatmap(freq_matrix, cols, save_path="../Src/structures/3modular_level_2/images/multienv/NoTears/ByGroup/edge_frequency_heatmap.png")

In [ ]:
dags = ProjectMethods.NoTearsLargeDataset(X_full=data, cols=multicausaldataset.columns.tolist(), save_point='../Src/structures/3modular_level_2/images/multienv/NoTears/fulldata', subset_size=20000000)
print(dags)

In [ ]:
img_path = '../Src/structures/3modular_level_2/images/multienv/NoTears/fulldata/dag.png'

# Load and show
img = mpimg.imread(img_path)
plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis('off')  # hide axes
plt.show()

<h2> Causal Discovery with extra info

In [ ]:
groups = [[1, 4, 6], [2, 7], [3, 8], [5]]

cols = ['Type_Cell', 'Age', 'Resistance', 'Temperature', 'Step_Index', 'Ambient_TemperatureC', 'VoltageV', 'CurrentA', 'Cycle_Index', 'TemperatureC_Cell_1', 'TemperatureC_Cell_2', 'TemperatureC_Cell_3', 'TemperatureC_Cell_4','CurrentA_Cell_1', 'CurrentA_Cell_2', 'CurrentA_Cell_3','CurrentA_Cell_4']
experiment_dataset_extra = ModularProcesses.getExperimentDataSetExtraInfo()
multicausaldataset_extra = CommonProcesses.multienvcausaldiscoverydataset(experiment_dataset_extra, cols, groups)
multicausaldataset_extra['Type_Cell_id'] = multicausaldataset_extra['Type_Cell'].astype('category').cat.codes
multicausaldataset_extra['Age_bin'] = multicausaldataset_extra['Age'].map({'Unaged': 0, 'Aged': 1})
model_features = ['Resistance', 'Temperature', 'Ambient_TemperatureC', 'VoltageV', 
                  'CurrentA', 'Cycle_Index', 'TemperatureC_Cell_1', 'TemperatureC_Cell_2', 'TemperatureC_Cell_3', 
                  'TemperatureC_Cell_4','CurrentA_Cell_1', 'CurrentA_Cell_2', 'CurrentA_Cell_3','CurrentA_Cell_4', 'Group', 'Type_Cell_id','Age_bin']
data_extra = multicausaldataset_extra[model_features].values
multicausaldatasetdivided_extra = dict(tuple(multicausaldataset_extra[model_features].groupby("Group")))


<h4> Per group

In [ ]:
dags = ProjectMethods.NoTearsByGroup(multicausaldatasetdivided_extra, '../Src/structures/3modular_level_2/images/multienvExtra/NoTears/ByGroup')
print(dags)

In [ ]:
dag_list, cols = ProjectMethods.load_all_dags("../Src/structures/3modular_level_2/images/multienvExtra/NoTears/ByGroup")           # from saved files
freq_matrix = ProjectMethods.dag_frequency_matrix(dag_list)
print("Matrix shape:", freq_matrix.shape)
print("Non-zero elements:", np.count_nonzero(freq_matrix))
print(freq_matrix)
ProjectMethods.plot_dag_frequency_heatmap(freq_matrix, cols, save_path="../Src/structures/3modular_level_2/images/multienvExtra/NoTears/ByGroup/edge_frequency_heatmap.png")

In [ ]:
dags = ProjectMethods.NoTearsLargeDataset(X_full=data_extra, cols=model_features, save_point='../Src/structures/3modular_level_2/images/multienvExtra/NoTears/fulldata', subset_size=20000000, n_repeats=10)
print(dags)

In [ ]:
img_path = '../Src/structures/3modular_level_2/images/multienvExtra/NoTears/fulldata/dag.png'

# Load and show
img = mpimg.imread(img_path)
plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis('off')  # hide axes
plt.show()

<h3> DoTears

In [3]:
groups = [[1, 4, 6], [2, 7], [3, 8], [5]]
cols = ['Ambient_TemperatureC', 'VoltageV', 'CurrentA', 'CycleIndex', 'TemperatureC_Cell_1', 'TemperatureC_Cell_2', 'TemperatureC_Cell_3', 'TemperatureC_Cell_4','CurrentA_Cell_1', 'CurrentA_Cell_2', 'CurrentA_Cell_3','CurrentA_Cell_4']
experiment_dataset = ModularProcesses.getLongCellDatasetModular()
multicausaldataset = CommonProcesses.envcausaldiscoverydataset(experiment_dataset, cols, groups)

In [4]:
print(multicausaldataset)

{'0':          Ambient_TemperatureC  VoltageV  CurrentA  CycleIndex  \
0                   19.451006  3.572477       0.0           1   
1                   19.451006  3.572603       0.0           1   
2                   19.561859  3.572414       0.0           1   
3                   19.490467  3.571972       0.0           1   
4                   19.492714  3.572288       0.0           1   
...                       ...       ...       ...         ...   
2390534             40.339209  2.977739       0.0           1   
2390535             40.409821  2.977361       0.0           1   
2390536             40.463728  2.977298       0.0           1   
2390537             40.509055  2.977802       0.0           1   
2390538             40.509060  2.977865       0.0           1   

         TemperatureC_Cell_1  TemperatureC_Cell_2  TemperatureC_Cell_3  \
0                  19.970938            20.079184            20.310024   
1                  19.970938            20.079184            20.3

In [5]:
data = multicausaldataset
data_for_dotears = {
    "obs": data["0"].values,
    "4_0": data["1"].values,
    "4_1": data["2"].values,
    "4_2": data["3"].values
}
for k, v in data_for_dotears.items():
    print(k, v.shape)

intervention_targets = {
    "0": 4,
    "1": 4,
    "2": 4,

}
w_threshold = 0.02

obs (977525, 12)
4_0 (851616, 12)
4_1 (324326, 12)
4_2 (249502, 12)


In [6]:
base_dir = "../Src/structures/3modular_level_2/images/multienv/DoTears/fulldata/"
for scaled in [False, True]:
    for lambda1 in [0.05, 0.1, 0.2]:
        model = dotears.DOTEARS(
            data_for_dotears,
            lambda1=lambda1,
            scaled=scaled,
            w_threshold=0.02
        )
        W = model.fit()
        print(f"scaled={scaled}, lambda={lambda1}")
        dag = (np.abs(W) > w_threshold).astype(int)
        # --- directory for group
        save_point = f"{base_dir}/{scaled}/{lambda1}"
        os.makedirs(save_point, exist_ok=True)
        ProjectMethods.makeAndSaveGraph(multicausaldataset["1"].columns.tolist(), dag, save_point)

scaled=False, lambda=0.05
scaled=False, lambda=0.1
scaled=False, lambda=0.2
scaled=True, lambda=0.05
scaled=True, lambda=0.1
scaled=True, lambda=0.2
